# Tree Models
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
Basic decision tree models are commonly used for classification.  They generate easy to interpret rules that determine probabilities of class membership.  Regression trees are less commonly used (as a standalone model) and are used for a continuous target.  There is math to build the rules, but the end result is easy to understand and interpret.


# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data - Universal Bank

To practice decision trees, we need a dataset that has a categorical target.  The Universal Bank dataset can be used to predict if a customer will accept an offer of a personal loan.  Each row represents a bank customer and the target variable is called Personal Loan.

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 7/UniversalBank.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 5000 rows and 14 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# list the columns with the data types
print(df.info())

Looks like a mix of categorical and continuous predictors.  We should  change our target variable to 1s and 0s.  And let's make dummy variables for Education.  We should drop Zip Code because it has too many categories.  

In [ ]:
# Convert 'Personal Loan' to numerical (Yes=1, No=0) using map
df['Personal Loan'] = df['Personal Loan'].map({'Yes': 1, 'No': 0})

# Display the first few rows and unique values to see the changes in the target variable
print(df.head())
print("\nUnique values after conversion:")
print(df['Personal Loan'].unique())

In [ ]:
# Look at counts of 0 and 1 values for Personal Loan to check our work
personal_loan_counts = df['Personal Loan'].value_counts()
print("Counts of Personal Loan (0s and 1s):")
print(personal_loan_counts)

In [ ]:
# Drop the 'ZIP Code' column
df = df.drop('ZIP Code', axis=1)

In [ ]:
# Create dummy variables for 'Education'
df = pd.get_dummies(df, columns=['Education'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Now let's partition the data to get it ready for modeling. We define Personal Loan as the target variable and the rest of the columns as our predictor variables.  We will do a 50/30/20 split.  In the code we first separate out the 20% for test, then we split the remaining portion into training and validation.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Personal Loan'
# Define features by dropping the target variable and the 'ID' column
features = df.drop([target, 'ID'], axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine the features and target for plotting
train_data = X_train.copy()
train_data['Personal Loan'] = y_train

val_data = X_val.copy()
val_data['Personal Loan'] = y_val

test_data = X_test.copy()
test_data['Personal Loan'] = y_test

# Plot the distribution of 'Personal Loan' in each partition
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

sns.countplot(data=train_data, x='Personal Loan', ax=axs[0])
axs[0].set_title('Training Set - Personal Loan Distribution')
axs[0].set_xlabel('Personal Loan')
axs[0].set_ylabel('Count')
axs[0].set_xticks([0, 1])
axs[0].set_xticklabels(['No', 'Yes'])

sns.countplot(data=val_data, x='Personal Loan', ax=axs[1])
axs[1].set_title('Validation Set - Personal Loan Distribution')
axs[1].set_xlabel('Personal Loan')
axs[1].set_ylabel('Count')
axs[1].set_xticks([0, 1])
axs[1].set_xticklabels(['No', 'Yes'])


sns.countplot(data=test_data, x='Personal Loan', ax=axs[2])
axs[2].set_title('Test Set - Personal Loan Distribution')
axs[2].set_xlabel('Personal Loan')
axs[2].set_ylabel('Count')
axs[2].set_xticks([0, 1])
axs[2].set_xticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

# Print counts and percentages for each partition
print("Training set Personal Loan counts and percentages:")
train_counts = train_data['Personal Loan'].value_counts()
train_percentages = train_data['Personal Loan'].value_counts(normalize=True) * 100
print(train_counts)
print(train_percentages.round(2)) # Round percentages to 2 decimal places


print("\nValidation set Personal Loan counts and percentages:")
val_counts = val_data['Personal Loan'].value_counts()
val_percentages = val_data['Personal Loan'].value_counts(normalize=True) * 100
print(val_counts)
print(val_percentages.round(2)) # Round percentages to 2 decimal places


print("\nTest set Personal Loan counts and percentages:")
test_counts = test_data['Personal Loan'].value_counts()
test_percentages = test_data['Personal Loan'].value_counts(normalize=True) * 100
print(test_counts)
print(test_percentages.round(2)) # Round percentages to 2 decimal places

# Decision Trees - Manual Math

We will start with forming a decision tree model manually to step through the logic in detail.

For a single variable, Income, we can calculate the Gini impurity if we split at a hardcoded value of 50.

In [ ]:
# Manual calculation of the first split in a decision tree

# Calculate the overall Gini impurity of the target variable
total_instances = len(y_train)
personal_loan_counts = y_train.value_counts()
gini_personal_loan = 1 - sum((count / total_instances) ** 2 for count in personal_loan_counts)

print(f"Overall Gini Impurity of Personal Loan: {gini_personal_loan:.4f}")

# Let's consider a potential split, for example, by 'Income' at a certain threshold
# We need to find potential split points. A common approach is to consider the midpoints
# between sorted unique values. For simplicity, let's pick an arbitrary threshold for now.
income_threshold = 50 # Example threshold

# Split the training data based on the income threshold
left_split_indices = X_train[X_train['Income'] <= income_threshold].index
right_split_indices = X_train[X_train['Income'] > income_threshold].index

y_train_left_split = y_train.loc[left_split_indices]
y_train_right_split = y_train.loc[right_split_indices]

# Calculate Gini impurity for the left split (Income <= 50)
total_left = len(y_train_left_split)
if total_left > 0:
    left_counts = y_train_left_split.value_counts()
    gini_left = 1 - sum((count / total_left) ** 2 for count in left_counts)
else:
    gini_left = 0 # If no instances in this split, impurity is 0

# Calculate Gini impurity for the right split (Income > 50)
total_right = len(y_train_right_split)
if total_right > 0:
    right_counts = y_train_right_split.value_counts()
    gini_right = 1 - sum((count / total_right) ** 2 for count in right_counts)
else:
    gini_right = 0 # If no instances in this split, impurity is 0


# Calculate the weighted average Gini impurity after the split
weighted_gini_split = (total_left / total_instances) * gini_left + (total_right / total_instances) * gini_right

# Calculate the Information Gain (reduction in Gini impurity)
information_gain = gini_personal_loan - weighted_gini_split

print(f"\nSplit based on Income <= {income_threshold}:")
print(f"  Gini Impurity (Income <= {income_threshold}): {gini_left:.4f} (n={total_left})")
print(f"  Gini Impurity (Income > {income_threshold}): {gini_right:.4f} (n={total_right})")
print(f"  Weighted Average Gini Impurity: {weighted_gini_split:.4f}")
print(f"  Information Gain: {information_gain:.4f}")

# To find the best split, you would repeat this process for all potential split points
# for each feature and select the split with the highest information gain.

This arbitrary split at 50 gives us a Gini impurity of 0.1519 and an information gain of 0.01.  As you can see, the manual approach is complicated and requires a lot of steps.  And this is just for one variable with one split value.  

Now let's improve that code to search for the optimal income to split on instead of the hard coded value of 50.  

In [ ]:
# Find potential split points for 'Income'
# Potential split points are the unique values of the feature
potential_splits = sorted(X_train['Income'].unique())

best_gini = float('inf')
best_threshold = None

# Calculate the overall Gini impurity (already calculated in the previous cell, but recalculating for clarity)
total_instances = len(y_train)
personal_loan_counts = y_train.value_counts()
gini_personal_loan = 1 - sum((count / total_instances) ** 2 for count in personal_loan_counts)


# Iterate through each potential split point
for threshold in potential_splits:
    # Split the training data based on the current threshold
    left_split_indices = X_train[X_train['Income'] <= threshold].index
    right_split_indices = X_train[X_train['Income'] > threshold].index

    y_train_left_split = y_train.loc[left_split_indices]
    y_train_right_split = y_train.loc[right_split_indices]

    # Calculate Gini impurity for the left split
    total_left = len(y_train_left_split)
    if total_left > 0:
        left_counts = y_train_left_split.value_counts()
        gini_left = 1 - sum((count / total_left) ** 2 for count in left_counts)
    else:
        gini_left = 0

    # Calculate Gini impurity for the right split
    total_right = len(y_train_right_split)
    if total_right > 0:
        right_counts = y_train_right_split.value_counts()
        gini_right = 1 - sum((count / total_right) ** 2 for count in right_counts)
    else:
        gini_right = 0

    # Calculate the weighted average Gini impurity after the split
    weighted_gini_split = (total_left / total_instances) * gini_left + (total_right / total_instances) * gini_right

    # Check if this split is better than the current best
    if weighted_gini_split < best_gini:
        best_gini = weighted_gini_split
        best_threshold = threshold

information_gain = gini_personal_loan - best_gini

print(f"Best split for Income:")
print(f"  Threshold: {best_threshold}")
print(f"  Weighted Average Gini Impurity: {best_gini:.4f}")
print(f"  Information Gain: {information_gain:.4f}")

By splitting at 115 for income, the results have improved.  We now have a lower Gini impurity of 0.1188.  Remember, when it comes to classification, lower impurity is better.  And we have a higher information gain of 0.0430.  This is also good because the split at this income level provides better differentiation than the arbitrary value of 50.  

Let's scale the code once more so that it now searches across all values of all variables to identify the best variable and value for the first split.

In [ ]:
# Calculate the overall Gini impurity (from previous calculation)
total_instances = len(y_train)
personal_loan_counts = y_train.value_counts()
gini_personal_loan = 1 - sum((count / total_instances) ** 2 for count in personal_loan_counts)

print(f"Overall Gini Impurity of Personal Loan: {gini_personal_loan:.4f}\n")

# Create a list to store the results for each feature
results = []

# Iterate through all predictor variables
for feature in X_train.columns:
    # Find potential split points for the current feature
    # For numerical features, use unique values. For categorical (dummy) features, the split is between 0 and 1.
    if X_train[feature].dtype in ['int64', 'float64']:
        potential_splits = sorted(X_train[feature].unique())
    else: # Assuming boolean/dummy variables
        potential_splits = [0.5] # Split between False (0) and True (1)

    best_gini = float('inf')
    best_threshold = None

    # Iterate through each potential split point for the current feature
    for threshold in potential_splits:
        # Split the training data based on the current threshold
        left_split_indices = X_train[X_train[feature] <= threshold].index
        right_split_indices = X_train[X_train[feature] > threshold].index

        y_train_left_split = y_train.loc[left_split_indices]
        y_train_right_split = y_train.loc[right_split_indices]

        # Calculate Gini impurity for the left split
        total_left = len(y_train_left_split)
        if total_left > 0:
            left_counts = y_train_left_split.value_counts()
            gini_left = 1 - sum((count / total_left) ** 2 for count in left_counts)
        else:
            gini_left = 0

        # Calculate Gini impurity for the right split
        total_right = len(y_train_right_split)
        if total_right > 0:
            right_counts = y_train_right_split.value_counts()
            gini_right = 1 - sum((count / total_right) ** 2 for count in right_counts)
        else:
            gini_right = 0

        # Calculate the weighted average Gini impurity after the split
        weighted_gini_split = (total_left / total_instances) * gini_left + (total_right / total_instances) * gini_right

        # Check if this split is better than the current best
        if weighted_gini_split < best_gini:
            best_gini = weighted_gini_split
            best_threshold = threshold

    information_gain = gini_personal_loan - best_gini

    # Store the results for the current feature
    results.append({
        'Feature': feature,
        'Best Threshold': best_threshold,
        'Weighted Gini Impurity': best_gini,
        'Information Gain': information_gain
    })

# Convert the results to a pandas DataFrame and display
results_df = pd.DataFrame(results)
display(results_df.round(4)) # Display with 4 decimal places for clarity

After exploring all the variables, we can clearly see that the best first split of the tree is still on income at a value of 115.  This split provides the lowest Gini impurity and the highest information gain.  

So much code and we only figured out the first split!  To continue with the manual approach, we would subset the data onto rows where income is greater than 115 and rows where income is less than 115.  We would then repeat the logic ove for each subset to find the next best split.  And so on and so on...

Next we will use the concise code to call the model function which is a lot easier.

# Decision Trees

First we identify the model then fit it to the training data.  

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree Classifier
# We can start with default parameters or set some like max_depth for initial exploration
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model on the training data
dt_model.fit(X_train, y_train)

print("Decision Tree model trained successfully.")

Let's look at the tree we just built.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Set the figure size for better visualization
plt.figure(figsize=(20, 15))

# Plot the original decision tree
plot_tree(dt_model,
          feature_names=list(X_train.columns), # Use feature names from the training data
          class_names=['No Personal Loan', 'Personal Loan'], # Use meaningful class names
          filled=True, # Color nodes to indicate the majority class
          rounded=True, # Round node corners
          fontsize=10) # Adjust font size for readability

plt.title('Original Decision Tree Visualization')
plt.show()

In [ ]:
# The number of splits in a decision tree is the number of internal nodes.
# This is equal to the total number of nodes minus the number of leaf nodes.
# We can access the tree structure from the tuned_dt_model.tree_ attribute.

tree_ = dt_model.tree_
total_nodes = tree_.node_count
leaf_nodes = tree_.n_leaves

number_of_splits = total_nodes - leaf_nodes

print(f"The decision tree has {number_of_splits} splits (internal nodes).")
print(f"The decision tree has {leaf_nodes} leaves.")

That's a big tree! How well does it perform?

In [ ]:
from sklearn.metrics import confusion_matrix

# Predictions on training set
y_train_pred = dt_model.predict(X_train)
cm_train = confusion_matrix(y_train, y_train_pred)

# Predictions on validation set
y_val_pred = dt_model.predict(X_val)
cm_val = confusion_matrix(y_val, y_val_pred)

# Predictions on test set
y_test_pred = dt_model.predict(X_test)
cm_test = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix - Training Set:")
print(cm_train)

print("\nConfusion Matrix - Validation Set:")
print(cm_val)

print("\nConfusion Matrix - Test Set:")
print(cm_test)

Can you see the problem here?  We have zero FP and FN on training.  The model is perfect!  This is a sure sign that the model is overfit to training!

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calculate metrics for Training Set
accuracy_train = accuracy_score(y_train, y_train_pred)
precision_train = precision_score(y_train, y_train_pred)
recall_train = recall_score(y_train, y_train_pred)
f1_train = f1_score(y_train, y_train_pred)

print("Training Set Metrics:")
print(f"  Accuracy: {accuracy_train:.4f}")
print(f"  Precision: {precision_train:.4f}")
print(f"  Recall: {recall_train:.4f}")
print(f"  F1-Score: {f1_train:.4f}")

# Calculate metrics for Validation Set
accuracy_val = accuracy_score(y_val, y_val_pred)
precision_val = precision_score(y_val, y_val_pred)
recall_val = recall_score(y_val, y_val_pred)
f1_val = f1_score(y_val, y_val_pred)

print("\nValidation Set Metrics:")
print(f"  Accuracy: {accuracy_val:.4f}")
print(f"  Precision: {precision_val:.4f}")
print(f"  Recall: {recall_val:.4f}")
print(f"  F1-Score: {f1_val:.4f}")


# Calculate metrics for Test Set
accuracy_test = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test = recall_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)

print("\nTest Set Metrics:")
print(f"  Accuracy: {accuracy_test:.4f}")
print(f"  Precision: {precision_test:.4f}")
print(f"  Recall: {recall_test:.4f}")
print(f"  F1-Score: {f1_test:.4f}")

And our other performance metrics tell the same story.

## Hyperparameter Tuning

There are some settings we can play with to keep the tree size smaller to prevent overfitting - max_depth, min_samples_split, and min_samples_leaf.  We don't know what the best combination of the those values are, but we can define reasonable ranges and then search for the optimal setup.  

First we will define the range of values to try for each parameter.  

In [ ]:
# Define the parameter grid
param_grid = {
    'max_depth': range(2, 21),
    'min_samples_split': range(2, 21),
    'min_samples_leaf': range(1, 11)
}

print("Parameter grid defined:")
print(param_grid)

Now we need to try out those ranges in differing combinations.  The first run might look like this: <br>
max_depth = 2 <br>
min_samples_split = 2 <br>
min_samples_leaf = 1 <br>


Then the next might look like this: <br>
max_depth = 2 <br>
min_samples_split = 2 <br>
min_samples_leaf = 2 <br>


Each of the parameters is changed one at a time until all possible combinations have been tried.  This code below takes a while to run becasue that's 19x19x10 = 3610 combinations!

It then selects the combination that gives the best average F1 Score across the five folds used during cross validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Instantiate GridSearchCV
grid_search = GridSearchCV(dt_model, param_grid, cv=5, scoring='f1', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Print the best hyperparameters and best cross-validation score
print("Best hyperparameters found:", grid_search.best_params_)
print("Best cross-validation F1-score:", grid_search.best_score_)

With the best parameters selected, we will rerun the model using those parameters to build a new tree that hopefully is not longer overfitted.  

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Retrieve the best hyperparameters
best_params = grid_search.best_params_
print("Best hyperparameters:", best_params)

# Instantiate a new Decision Tree Classifier with the best hyperparameters
tuned_dt_model = DecisionTreeClassifier(**best_params, random_state=42)

# Train the tuned model on the complete training dataset
tuned_dt_model.fit(X_train, y_train)

# Make predictions on training set
y_train_pred_tuned = tuned_dt_model.predict(X_train)
cm_train_tuned = confusion_matrix(y_train, y_train_pred_tuned)

# Make predictions on validation set
y_val_pred_tuned = tuned_dt_model.predict(X_val)
cm_val_tuned = confusion_matrix(y_val, y_val_pred_tuned)

# Make predictions on test set
y_test_pred_tuned = tuned_dt_model.predict(X_test)
cm_test_tuned = confusion_matrix(y_test, y_test_pred_tuned)

print("\nConfusion Matrix - Tuned Training Set:")
print(cm_train_tuned)

print("\nConfusion Matrix - Tuned Validation Set:")
print(cm_val_tuned)

print("\nConfusion Matrix - Tuned Test Set:")
print(cm_test_tuned)

# Calculate metrics for Training Set
accuracy_train_tuned = accuracy_score(y_train, y_train_pred_tuned)
precision_train_tuned = precision_score(y_train, y_train_pred_tuned)
recall_train_tuned = recall_score(y_train, y_train_pred_tuned)
f1_train_tuned = f1_score(y_train, y_train_pred_tuned)

print("\nTuned Training Set Metrics:")
print(f"  Accuracy: {accuracy_train_tuned:.4f}")
print(f"  Precision: {precision_train_tuned:.4f}")
print(f"  Recall: {recall_train_tuned:.4f}")
print(f"  F1-Score: {f1_train_tuned:.4f}")

# Calculate metrics for Validation Set
accuracy_val_tuned = accuracy_score(y_val, y_val_pred_tuned)
precision_val_tuned = precision_score(y_val, y_val_pred_tuned)
recall_val_tuned = recall_score(y_val, y_val_pred_tuned)
f1_val_tuned = f1_score(y_val, y_val_pred_tuned)

print("\nTuned Validation Set Metrics:")
print(f"  Accuracy: {accuracy_val_tuned:.4f}")
print(f"  Precision: {precision_val_tuned:.4f}")
print(f"  Recall: {recall_val_tuned:.4f}")
print(f"  F1-Score: {f1_val_tuned:.4f}")


# Calculate metrics for Test Set
accuracy_test_tuned = accuracy_score(y_test, y_test_pred_tuned)
precision_test_tuned = precision_score(y_test, y_test_pred_tuned)
recall_test_tuned = recall_score(y_test, y_test_pred_tuned)
f1_test_tuned = f1_score(y_test, y_test_pred_tuned)

print("\nTuned Test Set Metrics:")
print(f"  Accuracy: {accuracy_test_tuned:.4f}")
print(f"  Precision: {precision_test_tuned:.4f}")
print(f"  Recall: {recall_test_tuned:.4f}")
print(f"  F1-Score: {f1_test_tuned:.4f}")

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Set the figure size for better visualization
plt.figure(figsize=(20, 15))

# Plot the tuned decision tree
plot_tree(tuned_dt_model,
          feature_names=list(X_train.columns), # Use feature names from the training data
          class_names=['No Personal Loan', 'Personal Loan'], # Use meaningful class names
          filled=True, # Color nodes to indicate the majority class
          rounded=True, # Round node corners
          fontsize=10) # Adjust font size for readability

plt.title('Tuned Decision Tree Visualization')
plt.show()

In [ ]:
# The number of splits in a decision tree is the number of internal nodes.
# This is equal to the total number of nodes minus the number of leaf nodes.
# We can access the tree structure from the tuned_dt_model.tree_ attribute.

tree_ = tuned_dt_model.tree_
total_nodes = tree_.node_count
leaf_nodes = tree_.n_leaves

number_of_splits = total_nodes - leaf_nodes

print(f"The tuned decision tree has {number_of_splits} splits (internal nodes).")
print(f"The tuned decision tree has {leaf_nodes} leaves.")

Now let's look at the final rules in a format that's easier to read and use in the future.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# Assuming 'tuned_dt_model' is your trained Decision Tree Classifier model
# Assuming 'X_train' is your training feature DataFrame (for feature names)

def get_decision_path_rules(tree, feature_names, node_id, current_rule, class_names):
    """Recursively gets the decision path rules and probabilities for each leaf node."""
    left_child = tree.children_left[node_id]
    right_child = tree.children_right[node_id]

    # If it's a leaf node
    if left_child == -1 and right_child == -1:
        # Get the class distribution (counts) and total samples at the leaf
        value = tree.value[node_id][0]
        total_samples_at_leaf = np.sum(value)

        # Calculate probabilities for each class
        probabilities = value / total_samples_at_leaf

        # Format the probabilities for printing
        prob_strings = [f"{class_names[i]}: {probabilities[i]:.4f}" for i in range(len(class_names))]
        prob_output = ", ".join(prob_strings)

        print(f"Rule: {current_rule} -> Probabilities: {prob_output}")
        return

    # If it's an internal node
    feature_index = tree.feature[node_id]
    threshold = tree.threshold[node_id]
    feature_name = feature_names[feature_index]

    # Rule for the left child
    left_rule = f"{current_rule} AND {feature_name} <= {threshold:.2f}" if current_rule else f"{feature_name} <= {threshold:.2f}"
    get_decision_path_rules(tree, feature_names, left_child, left_rule, class_names)

    # Rule for the right child
    right_rule = f"{current_rule} AND {feature_name} > {threshold:.2f}" if current_rule else f"{feature_name} > {threshold:.2f}"
    get_decision_path_rules(tree, feature_names, right_child, right_rule, class_names)

# Get the tree structure and feature names
tree_structure = tuned_dt_model.tree_
feature_names = list(X_train.columns)
class_names = ['No Personal Loan', 'Personal Loan'] # Define your class names

print("Logical rules and probabilities for each leaf of the tuned decision tree:")
# Start the recursion from the root node (node_id 0) with an empty initial rule
get_decision_path_rules(tree_structure, feature_names, 0, "", class_names)

That's still a lot of rules to make sense of.  To see how much each variable contributes to our prediction, it is helpful to look at the feature importance.  

In [ ]:
import pandas as pd

# Get the feature importances from the tuned decision tree model
feature_importances_tuned_dt = tuned_dt_model.feature_importances_

# Get the feature names from the training data (assuming X_train is consistent)
feature_names_dt = X_train.columns

# Create a pandas Series to easily view feature importances with their names
feature_importance_series_tuned_dt = pd.Series(feature_importances_tuned_dt, index=feature_names_dt)

# Sort the feature importances in descending order
sorted_feature_importances_tuned_dt = feature_importance_series_tuned_dt.sort_values(ascending=False)

# Print the sorted feature importances
print("Feature Importances for Tuned Decision Tree:")
print(sorted_feature_importances_tuned_dt)

Looks like Education and Income contribute the most.

#Load the Data - ToyotaCorolla1000

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 7/ToyotaCorolla1000.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 1000 rows and 10 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# Create dummy variables for 'Fuel Type'
df = pd.get_dummies(df, columns=['Fuel Type'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Before we model, we need to partition the data.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Price'
features = df.drop(target, axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

# Regression Trees - Manual Math

We will start with building our regression tree manually.  In the decision tree example we started with a single hard codeded example for the first split.  Since we already built up the logic to iterate through all variables to find the best variable and split point, let's start there this time.  

First we need to quantify how much variance there is overall in the training partition.  That gives us something to compare to for each potential split we make next.  

In [ ]:
# Calculate the overall variance of the target variable in the training set
overall_variance = y_train.var()

# Print the calculated overall variance
print(f"Overall Variance of Price: {overall_variance:.4f}")

Now we need to identify all the possible split points for each of the predictors.  

In [ ]:
# Create a dictionary to store potential split points for each feature
potential_splits_dict = {}

# Iterate through each column (feature) in the X_train DataFrame
for feature in X_train.columns:
    # Check the data type of the feature
    if X_train[feature].dtype in ['int64', 'float64']:
        # For numerical features, find unique values, sort them, and calculate midpoints
        unique_values = sorted(X_train[feature].unique())
        potential_splits = [(unique_values[i] + unique_values[i+1]) / 2 for i in range(len(unique_values) - 1)]
        potential_splits_dict[feature] = potential_splits
    elif X_train[feature].dtype == 'bool':
        # For boolean/dummy features, the split is typically at 0.5
        potential_splits_dict[feature] = [0.5]
    # Note: If there are other categorical types that haven't been dummified,
    # we would need to handle them differently (e.g., by considering all possible
    # groupings of categories). For this dataset, dummy variables handle categoricals.


# Display the potential split points found for each feature
for feature, splits in potential_splits_dict.items():
    print(f"Potential splits for '{feature}': {splits[:5]}...") # Print only the first few splits for brevity

Next we iterate through all the possible split points for each variable to find the best split.

In [ ]:
# Calculate total instances for the current training set
total_instances = len(y_train)
overall_best_reduction = -float('inf')
overall_best_feature = None
overall_best_threshold = None

# Calculate the overall variance (from previous calculation)
overall_variance = y_train.var()

# Create a list to store the results for each feature
results = []

# Iterate through all predictor variables
for feature, potential_splits in potential_splits_dict.items():

    best_reduction_for_feature = -float('inf') # Use -inf as we are maximizing reduction
    best_threshold_for_feature = None

    # Iterate through each potential split point for the current feature
    for threshold in potential_splits:
        # Split the training data based on the current threshold
        left_split_indices = X_train[X_train[feature] <= threshold].index
        right_split_indices = X_train[X_train[feature] > threshold].index

        y_train_left_split = y_train.loc[left_split_indices]
        y_train_right_split = y_train.loc[right_split_indices]

        total_left = len(y_train_left_split)
        if total_left > 0:
            variance_left = y_train_left_split.var()
        else:
            variance_left = 0

        total_right = len(y_train_right_split)
        if total_right > 0:
            variance_right = y_train_right_split.var()
        else:
            variance_right = 0

        # Handle cases where a split results in an empty set (variance is undefined/NaN)
        if total_left == 0 or total_right == 0:
            weighted_variance_split = float('inf') # Or handle as appropriate, e.g., skip this split
        else:
             weighted_variance_split = (total_left / total_instances) * variance_left + (total_right / total_instances) * variance_right


        reduction_in_variance = overall_variance - weighted_variance_split

        if reduction_in_variance > best_reduction_for_feature:
             best_reduction_for_feature = reduction_in_variance
             best_threshold_for_feature = threshold

    # Store the results for the current feature
    results.append({
        'Feature': feature,
        'Best Threshold': best_threshold_for_feature,
        'Variance Reduction': best_reduction_for_feature
    })

    # Update overall best if the best for this feature is better
    if best_reduction_for_feature > overall_best_reduction:
        overall_best_reduction = best_reduction_for_feature
        overall_best_feature = feature
        overall_best_threshold = best_threshold_for_feature

# Convert the results to a pandas DataFrame and display
results_df = pd.DataFrame(results)
display(results_df.round(4)) # Display with 4 decimal places for clarity


print(f"\nOverall best first split for regression tree:")
print(f"  Feature: {overall_best_feature}")
print(f"  Threshold: {overall_best_threshold}")
print(f"  Maximum Reduction in Variance: {overall_best_reduction:.4f}")

Looks like our first split should be on Age with a value of 32.5.  This will lead to the maximum reduction in variance.

Just like with our manual decision tree, the next step would be to subset the data into two parts - those cars with an age greater than 32.5 and those cars with an age less than 32.5.  We would repeat the logic above on the two subsets and find the best next split overall to continue to grow the tree.  

#Regression Trees

Now let's use our simpler functions to build our regression tree.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize the Decision Tree Regressor
# We can start with default parameters for an initial model
reg_tree_model = DecisionTreeRegressor(random_state=42)

# Train the model on the training data
reg_tree_model.fit(X_train, y_train)

print("Regression Tree model trained successfully.")

And let's look at the tree itself.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Set the figure size for better visualization
plt.figure(figsize=(20, 15))

# Plot the regression tree
plot_tree(reg_tree_model,
          feature_names=list(X_train.columns), # Use feature names from the training data
          filled=True, # Color nodes based on the predicted value
          rounded=True, # Round node corners
          fontsize=10) # Adjust font size for readability

plt.title('Regression Tree Visualization')
plt.show()

Oh my!  That's even bigger than our decision tree!

Let's take a look at this giant tree's performance.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Calculate the number of splits and leaves
tree_ = reg_tree_model.tree_
total_nodes = tree_.node_count
leaf_nodes = tree_.n_leaves
number_of_splits = total_nodes - leaf_nodes

print(f"The regression tree has {number_of_splits} splits (internal nodes).")
print(f"The regression tree has {leaf_nodes} leaves.")

# Make predictions on training set
y_train_pred_reg = reg_tree_model.predict(X_train)

# Make predictions on validation set
y_val_pred_reg = reg_tree_model.predict(X_val)

# Make predictions on test set
y_test_pred_reg = reg_tree_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_reg = np.sqrt(mean_squared_error(y_train, y_train_pred_reg))
r2_train_reg = r2_score(y_train, y_train_pred_reg)

print("\nRegression Tree Performance on Training Set:")
print(f"  RMSE: {rmse_train_reg:.4f}")
print(f"  R-squared: {r2_train_reg:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_reg = np.sqrt(mean_squared_error(y_val, y_val_pred_reg))
r2_val_reg = r2_score(y_val, y_val_pred_reg)

print("\nRegression Tree Performance on Validation Set:")
print(f"  RMSE: {rmse_val_reg:.4f}")
print(f"  R-squared: {r2_val_reg:.4f}")

# Calculate and print metrics for Test Set
rmse_test_reg = np.sqrt(mean_squared_error(y_test, y_test_pred_reg))
r2_test_reg = r2_score(y_test, y_test_pred_reg)

print("\nRegression Tree Performance on Test Set:")
print(f"  RMSE: {rmse_test_reg:.4f}")
print(f"  R-squared: {r2_test_reg:.4f}")

Yep, that's definitely overfit.  Zero error and perfect R sqared.  And with 430 splits, let's definitely try some hyperparameter tuning!

## Hyperparameter Tuning

Time to make this tree smaller and fix the overfitting.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, make_scorer

# Define the parameter grid
param_grid_reg = {
    'max_depth': range(2, 21),
    'min_samples_split': range(2, 21),
    'min_samples_leaf': range(1, 11)
}

# Create a scorer for negative mean squared error (GridSearchCV maximizes scores)
neg_mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Instantiate GridSearchCV
# Use the original reg_tree_model as the estimator
grid_search_reg = GridSearchCV(reg_tree_model, param_grid_reg, cv=5, scoring=neg_mse_scorer, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_reg.fit(X_train, y_train)

# Print the best hyperparameters and best cross-validation score
print("Best hyperparameters found:", grid_search_reg.best_params_)
# Convert the best score back to positive MSE or RMSE for easier interpretation
best_mse = -grid_search_reg.best_score_
best_rmse = np.sqrt(best_mse)
print(f"Best cross-validation RMSE: {best_rmse:.4f}")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Retrieve the best hyperparameters for the regression tree
best_params_reg = grid_search_reg.best_params_
print("Best hyperparameters for Regression Tree:", best_params_reg)

# Instantiate a new Decision Tree Regressor with the best hyperparameters
tuned_reg_tree_model = DecisionTreeRegressor(**best_params_reg, random_state=42)

# Train the tuned model on the complete training dataset
tuned_reg_tree_model.fit(X_train, y_train)

# Calculate the number of splits and leaves for the tuned tree
tree_tuned_reg = tuned_reg_tree_model.tree_
total_nodes_tuned_reg = tree_tuned_reg.node_count
leaf_nodes_tuned_reg = tree_tuned_reg.n_leaves
number_of_splits_tuned_reg = total_nodes_tuned_reg - leaf_nodes_tuned_reg

print(f"\nTuned Regression Tree:")
print(f"  Number of splits (internal nodes): {number_of_splits_tuned_reg}")
print(f"  Number of leaves: {leaf_nodes_tuned_reg}")


# Make predictions on training set
y_train_pred_tuned_reg = tuned_reg_tree_model.predict(X_train)

# Make predictions on validation set
y_val_pred_tuned_reg = tuned_reg_tree_model.predict(X_val)

# Make predictions on test set
y_test_pred_tuned_reg = tuned_reg_tree_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_tuned_reg = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned_reg))
r2_train_tuned_reg = r2_score(y_train, y_train_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Training Set:")
print(f"  RMSE: {rmse_train_tuned_reg:.4f}")
print(f"  R-squared: {r2_train_tuned_reg:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_tuned_reg = np.sqrt(mean_squared_error(y_val, y_val_pred_tuned_reg))
r2_val_tuned_reg = r2_score(y_val, y_val_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Validation Set:")
print(f"  RMSE: {rmse_val_tuned_reg:.4f}")
print(f"  R-squared: {r2_val_tuned_reg:.4f}")


# Calculate and print metrics for Test Set
rmse_test_tuned_reg = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned_reg))
r2_test_tuned_reg = r2_score(y_test, y_test_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Test Set:")
print(f"  RMSE: {rmse_test_tuned_reg:.4f}")
print(f"  R-squared: {r2_test_tuned_reg:.4f}")

Now we have just 41 splits, so a much smaller model.  And it isn't quite as overfit as before.  

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Set the figure size for better visualization
plt.figure(figsize=(20, 15))

# Plot the tuned regression tree
plot_tree(tuned_reg_tree_model,
          feature_names=list(X_train.columns), # Use feature names from the training data
          filled=True, # Color nodes based on the predicted value
          rounded=True, # Round node corners
          fontsize=10) # Adjust font size for readability

plt.title('Tuned Regression Tree Visualization')
plt.show()

This looks better.  Now let's see the rules.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# Assuming 'tuned_reg_tree_model' is your trained Decision Tree Regressor model
# Assuming 'X_train' is your training feature DataFrame (for feature names)

def get_regression_path_rules(tree, feature_names, node_id, current_rule):
    """Recursively gets the decision path rules and predicted values for each leaf node in a regression tree."""
    left_child = tree.children_left[node_id]
    right_child = tree.children_right[node_id]

    # If it's a leaf node
    if left_child == -1 and right_child == -1:
        # Get the predicted value (mean of the target variable) at the leaf
        predicted_value = tree.value[node_id][0][0]
        print(f"Rule: {current_rule} -> Predicted Value: {predicted_value:.4f}")
        return

    # If it's an internal node
    feature_index = tree.feature[node_id]
    threshold = tree.threshold[node_id]
    feature_name = feature_names[feature_index]

    # Rule for the left child
    left_rule = f"{current_rule} AND {feature_name} <= {threshold:.2f}" if current_rule else f"{feature_name} <= {threshold:.2f}"
    get_regression_path_rules(tree, feature_names, left_child, left_rule)

    # Rule for the right child
    right_rule = f"{current_rule} AND {feature_name} > {threshold:.2f}" if current_rule else f"{feature_name} > {threshold:.2f}"
    get_regression_path_rules(tree, feature_names, right_child, right_rule)

# Get the tree structure and feature names
tree_structure_reg = tuned_reg_tree_model.tree_
feature_names_reg = list(X_train.columns)

print("Logical rules and predicted values for each leaf of the tuned regression tree:")
# Start the recursion from the root node (node_id 0) with an empty initial rule
get_regression_path_rules(tree_structure_reg, feature_names_reg, 0, "")

And lastly, the feature importance.

In [ ]:
import pandas as pd

# Get the feature importances from the tuned regression tree model
feature_importances = tuned_reg_tree_model.feature_importances_

# Get the feature names from the training data
feature_names = X_train.columns

# Create a pandas Series to easily view feature importances with their names
feature_importance_series = pd.Series(feature_importances, index=feature_names)

# Sort the feature importances in descending order
sorted_feature_importances = feature_importance_series.sort_values(ascending=False)

# Print the sorted feature importances
print("Feature Importances for Tuned Regression Tree:")
print(sorted_feature_importances)